In [2]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

In [4]:
import sys
print(sys.executable)

/workspaces/llm-zoomcamp-2026-code/.venv/bin/python


In [1]:
from embedder import Embedder

embed = Embedder()

2026-06-27 20:42:27.021377863 [W:onnxruntime:Default, device_discovery.cc:133 GetPciBusId] Skipping pci_bus_id for PCI path at "/sys/devices/LNXSYSTM:00/LNXSYBUS:00/PNP0A03:00/device:07/VMBUS:01/5620e0c7-8062-4dce-aeb7-520c7ef76171" because filename "5620e0c7-8062-4dce-aeb7-520c7ef76171" did not match expected pattern of [0-9a-f]+:[0-9a-f]+:[0-9a-f]+[.][0-9a-f]+


In [3]:
q1="How does approximate nearest neighbor search work?"
v1 = embed.encode(q1)

In [4]:
v1[0]

np.float64(-0.02058203437252893)

The question 1 answer is -0.02 approximately

<h2 > question2 </h2>

In [5]:
target_filename = "02-vector-search/lessons/07-sqlitesearch-vector.md"

page = next(
    doc for doc in documents
    if doc["filename"] == target_filename
)



In [6]:
page

{'content': '# Vector Search with sqlitesearch\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=csxKescwJYM&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous section we used minsearch for vector search.\n\nIt works, but it has three problems:\n\n- It rebuilds the index on every startup\n- It keeps everything in memory\n- It searches by brute force\n\n\nWith text search we never felt these. Indexing was fast because we\ndidn\'t embed anything. With vector search, indexing runs a neural\nnetwork over every document, so it takes a minute on our dataset.\nKeeping everything in memory is fine here, but a larger dataset would\nneed too much space.\n\nThe third problem is brute-force search. For every query we compare the\nquery vector against every single document. With 1,000 documents this is\nfine, probably even faster than anything smarter. But as the dataset\ngrows past 10,000 or so, it gets slow, and we\'ll want an approximate\nmethod instead.\n\nWhat we\'ve done 

In [7]:
page_vector = embed.encode(page["content"])

score = page_vector.dot(v1)

score

np.float64(0.36107027225589694)

question 2 's answer is approximately 0.36

<h2> Question 3 </h2>

In [8]:
from gitsource import chunk_documents
chunks = chunk_documents(documents, size=2000, step=1000)

In [9]:
texts = [chunk["content"] for chunk in chunks]

vectors = embed.encode_batch(texts)
# vectors is a list/array of embeddings. Each embedding has 384 numbers.

array([[-0.08756474,  0.0183638 , -0.08122425, ...,  0.0305382 ,
        -0.02172767,  0.03277498],
       [ 0.02436192, -0.10619479,  0.03307313, ...,  0.01430086,
        -0.0012554 ,  0.04325694],
       [-0.01780481,  0.03103092,  0.00856104, ...,  0.02220219,
        -0.03375532,  0.04288225],
       ...,
       [ 0.00980345,  0.04912257,  0.01207493, ..., -0.09453998,
        -0.06321277,  0.04775796],
       [-0.03622024,  0.06821856, -0.01540895, ..., -0.00271633,
         0.01875558,  0.01007469],
       [-0.02975661, -0.00552575, -0.03531848, ...,  0.01044234,
         0.02297965, -0.01966068]], shape=(295, 384))

In [11]:
len(vectors[0])

384

In [13]:
import numpy as np
X = np.array(vectors)

X.shape

(295, 384)

In [15]:
scores = X.dot(v1)

scores.shape

(295,)

In [16]:
best_idx = scores.argmax()

best_chunk = chunks[best_idx]

best_chunk["filename"]

'02-vector-search/lessons/07-sqlitesearch-vector.md'

In [17]:
print(best_chunk["filename"])
print(best_chunk["start"])
print(best_chunk["content"][:1000])
print(scores[best_idx])

02-vector-search/lessons/07-sqlitesearch-vector.md
1000
rch. We score
the query against every document and pick the top ones. It always finds
the true top matches, but it pays for that by touching everything.

Approximate nearest neighbor (ANN) search takes a shortcut. Instead of
comparing against everything, it first narrows down to a region of
likely matches. Then it scores only within that region. It may miss the
absolute best match, but the results are still good and it's much
faster.

```text
NN (exact):    compare query against ALL documents -> top 5
ANN (approx):  narrow down to a region -> compare within region -> top 5
```

## sqlitesearch

sqlitesearch is the persistent sibling of minsearch, and it solves both
problems at once.

We already used it in module 1 for persistent text search. It also does
vector search through its `VectorSearchIndex` class. It stores vectors
in SQLite, a real on-disk database, and uses ANN strategies for
retrieval. Because the data lives on disk, o

'02-vector-search/lessons/07-sqlitesearch-vector.md'

<h2> Question 4 </h2>

In [18]:
from minsearch import VectorSearch

vindex = VectorSearch()
vindex.fit(X, chunks)

In [19]:
query = "What metric do we use to evaluate a search engine?"

In [20]:
query_vector = embed.encode(query)

results = vindex.search(query_vector, num_results=5)

In [21]:
results[0]

{'start': 0,
 'content': "# Search Evaluation Metrics\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=TuirMy3Pdbk&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson, we computed relevance lists for search results.\nWe can turn those lists into metrics.\n\n## Hit Rate\n\nHit Rate (also called Recall@k) measures the fraction of queries where\nthe correct document appears anywhere in the results:\n\n```python\nexample = [\n    [1, 0, 0, 0, 0],\n    [0, 1, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [0, 0, 0, 0, 0],\n    [0, 1, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [0, 0, 1, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n]\n```\n\nEach line is one query. If a line contains `1`, search found the\ncorrect document somewhere in the top 5 results. If the line contains\nonly zeros, search did not find the correct document.\n\nIn our setup

Question 4's answer is 04-evaluation/lessons/05-search-metrics.md

<h2> Question 5 </h2>

In [22]:
q5 = "How do I store vectors in PostgreSQL?"

In [23]:
from minsearch import Index

In [24]:
tindex = Index(
    text_fields=["content"],
    keyword_fields=[]
)

tindex.fit(chunks)

Vector Search

In [29]:
query_vector = embed.encode(q5)

vector_results = vindex.search(query_vector, num_results=5)

Text Search

In [30]:
text_results = tindex.search(
    q5,
    num_results=5
)

In [31]:
print("VECTOR RESULTS")
for result in vector_results:
    print(result["filename"], result["start"])

print("\nTEXT RESULTS")
for result in text_results:
    print(result["filename"], result["start"])

VECTOR RESULTS
02-vector-search/lessons/08-pgvector.md 0
02-vector-search/lessons/08-pgvector.md 3000
03-orchestration/lessons/05-rag.md 2000
02-vector-search/lessons/08-pgvector.md 1000
02-vector-search/lessons/08-pgvector.md 2000

TEXT RESULTS
02-vector-search/lessons/02-embeddings.md 4000
03-orchestration/lessons/05-rag.md 1000
02-vector-search/lessons/01-intro.md 0
03-orchestration/lessons/05-rag.md 0
02-vector-search/lessons/01-intro.md 1000


In [32]:
vector_files = {result["filename"] for result in vector_results}
text_files = {result["filename"] for result in text_results}

vector_only = vector_files - text_files

vector_only

{'02-vector-search/lessons/08-pgvector.md'}

vindex was fitted with X, the matrix of embeddings.
tindex was fitted with text chunks and indexes their words.

# semantic / meaning-based
text → embedding → vector search

# word-based
text → text search

<h2>  Question 6 Reciprocal Rank Fusion (RRF) - Hybrid Search </h2>

It ignores the raw scores from each method, which live on different scales and aren't directly comparable. Instead, it looks only at the position of each document in each list.

In [33]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [34]:
q6 = "How do I give the model access to tools?"

In [36]:
text_results6 = tindex.search(
    q6,
    num_results=5
)

In [35]:
query_vector6 = embed.encode(q6)

vector_results6 = vindex.search(query_vector6, num_results=5)

In [37]:
results = rrf([vector_results6, text_results6])

In [43]:
results[0]["filename"]

'01-agentic-rag/lessons/13-function-calling.md'